# [실습] 오픈 모델 전환

이번 실습부터는, 오픈 모델로 에이전트를 구현해 보겠습니다.  
vLLM으로 Qwen3.6-35B-A3B를 서빙하고, 오픈 모델를 에이전트 하네스에 연결합니다.

## 학습 목표

- vLLM으로 오픈 모델을 OpenAI 호환 엔드포인트로 서빙
- `PLATFORM='vllm'`으로 에이전트를 vLLM 서버에서 실행
- 오픈 모델의 도구 호출과 `response_format` 기반 구조화 응답 확인
- `ChatOpenAIWithReasoning`으로 reasoning 필드를 분리 출력

## 환경 설정

vLLM은 OpenAI 호환 서버로 동작합니다. (https://docs.vllm.ai/en/stable/serving/openai_compatible_server/)

vLLM은 버전에 따라 필요한 PyTorch/CUDA 버전이 다릅니다.
기존 환경과의 충돌을 피하기 위해 uvx로 격리된 가상 환경에서 실행합니다.


### uv

uv는 Python 패키지 설치와 가상 환경 관리를 빠르게 처리하는 도구입니다. (https://docs.astral.sh/uv/)

특히 uvx는 명령 실행 시 임시 격리 환경을 만들어, 기존 프로젝트 환경을 변화시키지 않고 vLLM 같은 도구를 실행할 수 있습니다.

In [ ]:
%pip install openai langchain langchain-openai langchain-ollama langchain-community -q

## 모델 다운로드

HuggingFace CLI로 모델을 다운로드합니다. 새 터미널에서 아래 커맨드를 실행하세요.

In [ ]:
# 아래 커맨드를 복사해 터미널에서 실행
# cd /workspace/lab && hf download QuantTrio/Qwen3.6-35B-A3B-AWQ --local-dir ./outputs/models/Qwen3.6-35B-A3B-AWQ

## vLLM 서빙

다운로드한 모델 위치를 아래 커맨드에 넣고 터미널에서 실행합니다.
단일 A40 48GB 기준이며, 128k 컨텍스트로 실행합니다.

### 도구 호출 지원
```
uvx --python 3.12 --from vllm==0.19.1 vllm serve outputs/models/Qwen3.6-35B-A3B-AWQ \
  --port 8000 --max-model-len 131072 --gpu-memory-utilization 0.7 \
  --reasoning-parser qwen3 --quantization moe_wna16 --language-model-only \
  --enable-auto-tool-choice --tool-call-parser qwen3_coder \
  --speculative-config '{"method":"mtp","num_speculative_tokens":2}' \
  --served-model-name Qwen3.6-35B-A3B-AWQ
```

주요 인자
- `--max-model-len 131072`: 최대 컨텍스트 길이 (128k)
- `--quantization moe_wna16`: MoE AWQ 양자화 커널
- `--reasoning-parser qwen3`: thinking 토큰을 reasoning 필드로 분리
- `--language-model-only`: 비전 인코더를 사용하지 않고 텍스트 전용으로 서빙
- `--enable-auto-tool-choice --tool-call-parser qwen3_coder`: 도구 호출을 chat template에 맞게 파싱
- `--speculative-config '{"method":"mtp","num_speculative_tokens":2}`: Multi-Token Prediction을 이용한 빠른 추론
- `--served-model-name Qwen3.6-35B-A3B-AWQ` : 모델의 서빙 시 사용할 이름



## 헬스체크와 호환 확인

서버가 실행 중이면 모델 목록을 확인합니다. 응답 구조는 OpenAI와 동일합니다.

In [ ]:
!curl http://0.0.0.0:8000/v1/models

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="token-abc123")

model_name = client.models.list().data[0].id
print(model_name)

completion = client.responses.create(model=model_name, input="안녕", temperature=0.7)
print(completion.output_text)

## LangChain 연결과 reasoning

vLLM의 --reasoning-parser qwen3는 thinking을 reasoning 필드로 분리해 반환합니다.  
하지만 LangChain의 ChatOpenAI는 이 필드를 무시하므로, reasoning을 보려면 ChatOpenAIWithReasoning을 씁니다.  
select_model의 vllm 경로는 이 클래스가 있으면 자동으로 사용합니다.

In [ ]:
# vLLM reasoning 필드를 읽는 클래스 다운로드
import urllib.request, os
url = "https://raw.githubusercontent.com/NotoriousH2/chat-openai-with-reasoning/master/chat_openai_with_reasoning.py"
urllib.request.urlretrieve(url, "chat_openai_with_reasoning.py")
print("다운로드 완료:", os.path.abspath("chat_openai_with_reasoning.py"))

## 플랫폼 선택

모델은 select_model의 load_model로 불러옵니다.  
이 노트북은 vLLM 서버를 실행해 두고 진행하므로 PLATFORM을 vllm으로 둡니다.  
vllm은 모델명을 None으로 두면 서버가 올린 모델을 자동 감지합니다.

In [ ]:
from select_model import load_model

PLATFORM = 'vllm'          # 'openai' | 'vllm' | 'ollama'

# vllm은 None이면 서버가 올린 모델을 자동 감지합니다.
MODEL_NAME = {'openai': 'gpt-5.2', 'vllm': None, 'ollama': 'qwen3.6:latest'}[PLATFORM]

VLLM_SAMPLING = dict(temperature=0.7, top_p=0.80, presence_penalty=1.5, extra_body={"top_k": 20, "min_p": 0.0, "repetition_penalty": 1.0}) if PLATFORM == 'vllm' else {}
model = load_model(MODEL_NAME, platform=PLATFORM, **VLLM_SAMPLING)
print(f'{PLATFORM} 모델 로드:', type(model).__name__)

In [ ]:
# enable_thinking으로 reasoning 출력을 선택합니다.
reasoning_llm = load_model(platform='vllm', enable_thinking=True, max_tokens=8192, temperature=1.0, top_p=0.95, presence_penalty=1.5, extra_body={"top_k": 20, "min_p": 0.0, "repetition_penalty": 1.0})

resp = reasoning_llm.invoke('vLLM의 장점을 한 문장으로?')
print('=== Reasoning ===')
print(resp.additional_kwargs.get('reasoning', '(없음)'))
print('\n=== Answer ===')
print(resp.text)

In [ ]:
# 스트리밍에서 reasoning과 content를 구분해 출력
in_reasoning = True
print('=== Reasoning ===')
for s in reasoning_llm.stream('vLLM과 Ollama의 차이를 한 문단으로?'):
    r = s.additional_kwargs.get('reasoning', '')
    c = s.text or ''
    if r:
        print(r, end='')
    if c:
        if in_reasoning:
            print('\n\n=== Answer ===')
            in_reasoning = False
        print(c, end='')

## 엔드포인트 스왑

vLLM은 OpenAI 호환 엔드포인트를 노출합니다.
앞 실습의 에이전트 코드에서 모델 로드 부분을 `load_model(platform='vllm')`로 바꿉니다.

아래에서 앞 실습과 같은 방식으로 도구 에이전트를 만들고, vLLM 서버 위에서 실행합니다.

도구 에이전트가 동작하려면 서버를 위의 도구 호출 지원 커맨드로 실행해야 합니다.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from datetime import datetime

@tool
def get_current_datetime() -> str:
    """현재 날짜와 시간을 반환합니다."""
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

agent = create_agent(
    model,
    tools=[get_current_datetime],
    system_prompt='도구를 사용하기 전 중간 과정을 간략히 설명하세요.',
)

result = agent.invoke({'messages': [{'role': 'user', 'content': '지금 날짜와 시간 알려줘.'}]})
print(result['messages'][-1].text)

In [ ]:
from stream_utils import stream_with_markdown

result = await stream_with_markdown(agent, '오늘 날짜를 알려주고, 오픈 모델을 직접 서빙할 때의 장점을 세 가지로 설명해줘.')

## 오픈 모델의 도구 호출

오픈 모델 에이전트에서는 응답에 구조화된 도구 호출이 포함되는지 확인합니다.

vLLM은 `auto-tool-choice`와 모델 chat template에 맞는 tool-call 파서를 함께 사용해야 도구 호출을 구조화해 반환합니다.
파서가 맞지 않으면 도구 호출이 raw 텍스트로 반환되어 에이전트의 도구 루프가 실행되지 않습니다.

## 구조화 응답 확인

라우팅 노드는 `response_format`을 받고 `structured_response`를 반환합니다.

`create_agent`에 Pydantic 스키마를 전달하고, 결과를 `structured_response`에서 확인합니다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Route(BaseModel):
    reasoning: str = Field(description='이 경로를 고른 한 문장 근거')
    destination: Literal['web_search', 'compute', 'chitchat']

router = create_agent(model, tools=[], response_format=Route)

out = router.invoke({'messages': [{'role': 'user', 'content': '작년 한 해 원달러 환율 추이를 찾아줘'}]})
print(out['structured_response'])

## build_agent로 배포 에이전트 만들기

배포 팩토리 build_agent.py는 platform 인자로 모델 플랫폼을 고릅니다. `platform='vllm'`을 주면 같은 에이전트가 vLLM 서버의 오픈 모델에서 동작합니다. 모델과 도구 구성은 build_agent가 맡으므로 Slack 봇은 build_agent()만 호출합니다.

In [ ]:
%%writefile build_agent.py
"""build_agent.py

에이전트를 한 곳에서 조립하는 팩토리입니다. 모델은 select_model로 고르고,
표준 도구(tools.py)에 MCP 서버 도구를 더해 에이전트를 만듭니다. 서비스 코드(예: Slack 봇)는
build_agent()만 호출해 같은 에이전트를 받습니다.

MCP 도구를 await로 받아오므로 build_agent는 async 함수입니다.
"""

from __future__ import annotations

import os

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from select_model import load_model
from tools import DEFAULT_TOOLS
from mcp_config import load_servers

DEFAULT_SYSTEM_PROMPT = (
    '당신은 유용한 AI 어시스턴트입니다. '
    '필요하면 도구를 사용하고, 도구를 실행하기 전 중간 과정을 간략히 설명하세요.'
)

# build_agent가 직접 연결하는 MCP 서버. mcp_servers.json의 stateless 서버를 읽는다.
# stateful 세션(Playwright 등)은 봇이 세션으로 열어 extra_tools로 넘긴다.
MCP_SERVERS = load_servers(stateful=False)


async def _auto_elicitation(mcp_context, params, context):
    """Slack MCP의 post_message가 채널을 되물을 때 기본 채널로 자동 응답합니다.

    봇이나 노트북에는 사용자에게 입력 폼을 표시할 방법이 없으므로, SLACK_CHANNEL_ID
    환경변수의 채널로 답합니다.
    """
    from mcp.types import ElicitResult

    channel = os.environ.get('SLACK_CHANNEL_ID', 'C_DRYRUN')
    return ElicitResult(action='accept', content={'channel_id': channel})


async def load_mcp_tools(servers=None):
    """MCP 서버에서 도구를 받아옵니다. 연결에 실패하면 빈 목록을 돌려줍니다."""
    servers = MCP_SERVERS if servers is None else servers
    if not servers:
        return []
    try:
        from langchain_mcp_adapters.callbacks import Callbacks

        client = MultiServerMCPClient(
            servers, callbacks=Callbacks(on_elicitation=_auto_elicitation)
        )
        return await client.get_tools()
    except Exception as e:
        print(f'[build_agent] MCP 도구 로드 실패, 표준 도구만 사용합니다: {e}')
        return []


async def build_agent(
    model_name=None,
    platform='openai',
    model=None,
    tools=None,
    extra_tools=None,
    system_prompt=None,
    checkpointer=None,
    mcp_servers=None,
    **model_kwargs,
):
    """현재까지 구성된 에이전트를 만들어 반환합니다.

    Args:
        model_name: 모델 이름. select_model.load_model로 전달됩니다.
        platform: 'openai' | 'vllm' | 'ollama'. load_model로 전달됩니다.
        model: 이미 만든 chat model. 주면 model_name/platform은 무시됩니다.
        tools: 표준 도구 대신 쓸 도구 목록. 생략 시 tools.py의 DEFAULT_TOOLS.
        extra_tools: 호출자가 직접 만든 도구를 추가로 붙입니다. 봇이 stateful
            세션(예: Playwright)에서 받은 도구를 넘길 때 씁니다.
        system_prompt: 시스템 프롬프트. 생략 시 기본값을 씁니다.
        checkpointer: 멀티턴 대화를 위한 체크포인터.
        mcp_servers: 연결할 MCP 서버 설정. 생략 시 MCP_SERVERS, {}면 MCP를 끕니다.
        model_kwargs: load_model로 전달되는 추가 인자 (reasoning_effort 등).
    """
    if model is None:
        model = load_model(model_name, platform=platform, **model_kwargs)
    base_tools = list(DEFAULT_TOOLS) if tools is None else list(tools)
    mcp_tools = await load_mcp_tools(mcp_servers)
    extra = list(extra_tools) if extra_tools else []
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT
    return create_agent(
        model,
        tools=base_tools + mcp_tools + extra,
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

build_agent.py를 import해 vLLM 플랫폼으로 에이전트를 만듭니다. `mcp_servers={}`로 MCP 연결 없이 tools.py의 표준 도구만 사용합니다.

In [ ]:
from build_agent import build_agent

# 플랫폼 선택 셀의 PLATFORM, MODEL_NAME으로 에이전트를 만든다.
agent = await build_agent(platform=PLATFORM, model_name=MODEL_NAME, mcp_servers={})

result = await agent.ainvoke({
    'messages': [{'role': 'user', 'content': '123 곱하기 47을 계산하고 결과를 한 문장으로 알려줘.'}]
})
print(result['messages'][-1].text)

## 35B-A3B의 서빙 특성

Qwen3.6-35B-A3B는 총 35B 파라미터 중 토큰마다 약 3B만 활성화하는 MoE입니다.
활성 파라미터가 작아 단일 A40의 짧은 컨텍스트 디코드 throughput은 약 90 tok/s입니다.
하이브리드 선형 어텐션으로 KV 캐시가 작아 단일 A40 48GB에 128k 컨텍스트가 들어갑니다.
이 특성 때문에 단일 A40에서 실습용 서빙이 가능합니다.

## 컨텍스트와 KV 관리

서버는 128k 윈도우로 실행 중이지만, 단일 GPU의 KV 예산은 한정되어 있습니다.
동시 요청이 늘거나 도구 출력이 길어지면 KV 사용량이 빠르게 증가합니다.

도구 출력과 이전 메시지 중 다음 호출에 필요한 정보만 남깁니다.

## 로컬에서 원격 호출

서빙 중인 모델은 외부에서 8000번 포트로 연결할 수 있습니다. 아래 base_url을 현재 클라우드 주소의 8000번 포트로 바꾸고 로컬 환경에서 실행해 보세요.

In [ ]:
from openai import OpenAI

# 현재 Pod의 8000번 포트 프록시 주소로 변경
base_url = "https://uuko0tf9fm17ay-8000.proxy.runpod.net/v1"
client = OpenAI(base_url=base_url, api_key="token-abc123")

print(client.models.list().data[0].id)

## 서버 종료

서버는 실행한 터미널에서 `CTRL+C`로 종료합니다.
터미널 창만 닫으면 프로세스가 남을 수 있습니다.

## 정리

- vLLM으로 오픈 모델을 OpenAI 호환 엔드포인트로 서빙
- `load_model(..., platform='vllm')`로 앞 실습 에이전트를 vLLM 서버에서 실행
- 도구 호출은 auto-tool-choice와 tool-call 파서로 구조화
- 라우팅 결과는 `response_format`과 `structured_response`로 확인